# Deploy

In [ ]:
import os
from difflib import SequenceMatcher

import pandas as pd
from dotenv import load_dotenv
from tqdm import trange

from lib.kicktipp import Bet, KicktippApi

load_dotenv()

KICKTIPP_USERNAME = os.getenv("KICKTIPP_USERNAME")
KICKTIPP_PASSWORD = os.getenv("KICKTIPP_PASSWORD")
KICKTIPP_COMMUNITY = os.getenv("KICKTIPP_COMMUNITY")

if not KICKTIPP_USERNAME or not KICKTIPP_PASSWORD or not KICKTIPP_COMMUNITY:
    raise ValueError(
        "Please set KICKTIPP_USERNAME, KICKTIPP_PASSWORD and "
        "KICKTIPP_COMMUNITY in your .env file"
    )

In [21]:
df_predictions = pd.read_pickle("data/predictions_poisson_2025.pkl")
df_predictions.head()

,match_day,season,league,datetime,home,away,home_goals,away_goals,home_goals_pred,away_goals_pred
id,,,,,,,,,,
77256,1,2025,bl1,2025-08-22 20:30:00,Bayern,Leipzig,6,0,2,1
77257,1,2025,bl1,2025-08-23 15:30:00,Leverkusen,Hoffenheim,1,2,2,1
77258,1,2025,bl1,2025-08-23 15:30:00,Frankfurt,Bremen,4,1,2,1
77259,1,2025,bl1,2025-08-23 15:30:00,Freiburg,Augsburg,1,3,1,0
77262,1,2025,bl1,2025-08-23 15:30:00,Union Berlin,Stuttgart,2,1,1,1


In [22]:
api = KicktippApi()
api.login(KICKTIPP_USERNAME, KICKTIPP_PASSWORD)

In [ ]:
# create mapping from kicktipp team names to prediction team names
bets_old = api.get_open_bets(KICKTIPP_COMMUNITY, 34)
kicktipp_team_names = [m.away_team for m in bets_old] + [m.home_team for m in bets_old]
prediction_team_names = set(
    df_predictions["home"].unique().tolist() + df_predictions["away"].unique().tolist()
)

kicktipp_to_prediction_team_map = {}
for kt in kicktipp_team_names:
    best_match = None
    best_ratio = 0.0
    for pt in prediction_team_names:
        ratio = SequenceMatcher(None, kt, pt).ratio()
        if ratio > best_ratio:
            best_ratio = ratio
            best_match = pt
    kicktipp_to_prediction_team_map[kt] = best_match

kicktipp_to_prediction_team_map

{'1. FC Köln': 'Köln',
 'Hamburger SV': 'Hamburger SV',
 'VfB Stuttgart': 'Stuttgart',
 '1899 Hoffenheim': 'Hoffenheim',
 'FSV Mainz 05': 'Mainz',
 'RB Leipzig': 'Leipzig',
 'FC Augsburg': 'Augsburg',
 'VfL Wolfsburg': 'Wolfsburg',
 'Borussia Dortmund': 'Dortmund',
 'FC Bayern München': 'Bayern',
 'Bayer 04 Leverkusen': 'Leverkusen',
 'Eintracht Frankfurt': 'Frankfurt',
 'Bor. Mönchengladbach': 'Gladbach',
 '1. FC Heidenheim 1846': 'Heidenheim',
 'SC Freiburg': 'Freiburg',
 '1. FC Union Berlin': 'Union Berlin',
 'FC St. Pauli': 'St. Pauli',
 'Werder Bremen': 'Bremen'}

In [ ]:
# submit bets for all match days
for match_day in trange(1, 35):
    bets_old = api.get_open_bets(KICKTIPP_COMMUNITY, match_day)
    bets_new = []
    for bet in bets_old:
        try:
            match = df_predictions[
                (
                    df_predictions["home"]
                    == kicktipp_to_prediction_team_map[bet.home_team]
                )
                & (
                    df_predictions["away"]
                    == kicktipp_to_prediction_team_map[bet.away_team]
                )
            ].iloc[0]
        except IndexError:
            print(f"Could not find prediction for {bet.home_team} vs {bet.away_team}")
            continue

        bets_new.append(
            Bet(
                match_day=bet.match_day,
                match_id=bet.match_id,
                home_team=bet.home_team,
                away_team=bet.away_team,
                home_bet=match["home_goals_pred"],
                away_bet=match["away_goals_pred"],
            )
        )

    api.submit_bets(KICKTIPP_COMMUNITY, bets_new)

100%|██████████| 34/34 [00:06<00:00,  5.24it/s]
